In [0]:
from pyspark.sql.functions import *

silver_txn = spark.table(
    "banking_catalog.banking_schema.silver_transactions"
)

silver_txn.groupBy("payment_format") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, False)

+--------------+-------+
|payment_format|count  |
+--------------+-------+
|Cheque        |1864331|
|Credit Card   |1323324|
|ACH           |600793 |
|Cash          |490891 |
|Reinvestment  |481056 |
|Wire          |171855 |
|Bitcoin       |145697 |
+--------------+-------+



In [0]:
gold_aml_df = (
    silver_txn
    .groupBy("payment_format")
    .agg(
        count("*").alias("total_transactions"),
        sum(
            when(col("is_laundering") == 1, 1)
            .otherwise(0)
        ).alias("laundering_transactions"),
        sum("amount_paid").alias("total_amount"),
        sum(
            when(
                col("is_laundering") == 1,
                col("amount_paid")
            ).otherwise(0)
        ).alias("laundering_amount")
    )
    .withColumn(
        "laundering_percentage",
        round(
            (col("laundering_transactions") * 100.0)
            / col("total_transactions"),
            2
        )
    )
)

In [0]:
display(
    gold_aml_df.orderBy(
        col("laundering_transactions").desc()
    )
)

payment_format,total_transactions,laundering_transactions,total_amount,laundering_amount,laundering_percentage
ACH,600793,4483,5706361955895.69,186633545939.59,0.75
Cheque,1864331,324,11379133080333.16,178520740.96,0.02
Credit Card,1323324,206,114842516515.60,5039261.53,0.02
Cash,490891,108,3612925484158.12,255396030.03,0.02
Bitcoin,145697,56,4504326.18,35.01,0.04
Reinvestment,481056,0,1248344677878.30,0.00,0.0
Wire,171855,0,838033641563.49,0.00,0.0


In [0]:
gold_aml_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.banking_schema.gold_aml_summary"
    )

In [0]:
spark.sql("""
SELECT *
FROM banking_catalog.banking_schema.gold_aml_summary
ORDER BY laundering_transactions DESC
""").show()

+--------------+------------------+-----------------------+-----------------+-----------------+---------------------+
|payment_format|total_transactions|laundering_transactions|     total_amount|laundering_amount|laundering_percentage|
+--------------+------------------+-----------------------+-----------------+-----------------+---------------------+
|           ACH|            600793|                   4483| 5706361955895.69|  186633545939.59|                 0.75|
|        Cheque|           1864331|                    324|11379133080333.16|     178520740.96|                 0.02|
|   Credit Card|           1323324|                    206|  114842516515.60|       5039261.53|                 0.02|
|          Cash|            490891|                    108| 3612925484158.12|     255396030.03|                 0.02|
|       Bitcoin|            145697|                     56|       4504326.18|            35.01|                 0.04|
|  Reinvestment|            481056|                     